##### Import libraries

In [173]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
import json
import unicodedata
import spacy
from spacy.matcher import PhraseMatcher
from spacy.tokens import Token

pd.set_option("display.max_columns", None)   # no column truncation
pd.set_option("display.max_rows", None)      # no row truncation
pd.set_option("display.width", None)         # no line-wrapping
pd.set_option("display.max_colwidth", None)

### Configuration and Functions used in the notebook

##### ACE Terms dictionary

In [212]:
# 1) Minimal ACE lexicon : Need to expand/refine taking into account "terminos.docx" file
# Dict keys: concept_id -> mapped from "terminos.docx" file
# Values: list of terms (lowercase), to be refined 
# Probably need to treat the accentuations.        
ACE_TERMS = {
    "ACE_SEXUAL_ABUSE": [
        "abuso sexual", "abusos sexuales", "violación", "agresión sexual", "tocamientos", "abuso sexual infantil",
        "maltrato sexual", "sexo oral", "penetración", "tocamientos", "voyeurismo", "exhibicionismo", "explotación",
        "explotación sexual", "víctima de agresión sexual"
    ],
    "ACE_PHYSICAL_ABUSE": [
        "maltrato físico", "golpes", "agresión física", "palizas", "violencia física",
        "malos tratos físicos", "abuso físico", "golpeado", "patada", "puñetazo", "azotes", 
        "agitar", "empujar", "arañar", "pellizcar", "morder", "estrangular", "torturar", "quemar", 
        "lanzar", "sacudir", "síndrome del bebé zarandeado", "síndrome de Munchausen",
        "víctima de abuso físico", "malos tratos"
    ],
    "ACE_EMOTIONAL_ABUSE": [
        "maltrato psicológico", "abuso emocional", "humillaciones", "insultos", "amenazas",
        "abuso psicológico", "malos tratos psicológicos", "maltrato emocional", "maltrato verbal",
        "víctima de abuso emocional", "víctima de abuso psicológico"
    ],
    "ACE_PHYSICAL_NEGLECT": [
        "abandono", "falta de cuidados",  "descuido"
    ],
    "ACE_EMOTIONAL_NEGLECT": [
        "desatención", # Refine with appropiate terms
    ],
    "ACE_UNSPECIFIED_NEGLECT": [
        "negligencia", "comportamiento negligente", "falta de cuidados",  
    ],
    "ACE_DOMESTIC_VIOLENCE_EXPOSURE": [
        "violencia intrafamiliar", "violencia doméstica", "violencia en casa", "malos tratos en el hogar",
        "testigo de violencia doméstica"
    ],
    "ACE_PARENT_SUBSTANCE": [
        "padre alcohólico", "madre alcohólica", "consumo de alcohol del padre", "drogadicción del padre",
        "consumo de drogas en casa"
    ],
    "ACE_PARENT_MENTAL_ILLNESS": [
        "madre con depresión", "padre con depresión", "trastorno mental del padre", "psicosis del padre"
    ],
    "ACE_PARENT_INCARCERATION": [
        "padre en prisión", "madre en prisión", "encarcelamiento del padre",
        "encarcelamiento de la madre"
    ],
    "ACE_PARENT_DIVORCE_SEPARATION": [
        "divorcio de los padres", "separación de los padres", "padres separados", "padres divorciados"
    ],
    "ACE_POVERTY_UNEMPLOYMENT": [
        "padres desempleados", "padres en paro", "padres sin empleo", 
        "padre desempleado", "padre en paro", "padre sin empleo", "padre no trabaja",
        "madre desempleada", "madre en paro", "madre sin empleo", "madre no trabaja", 
        "sin medios", "medios limitados", "medios económicos limitados" 
    ],
    "ACE_GENERAL_TRAUMA": [
        "adversidad precoz", "trauma precoz", "traumatismo precoz", "evento traumático", "evento adverso",
        "situación traumática", "vivencia traumática", "antecedente traumático", 
        "historia de trauma", "experiencia traumática", "suceso traumático",
        "violencia", "violento", "testigo de violencia", "víctima", "maltrato"
    ], 
}

##### Regex configuration

In [254]:

# Lightweight negation cues. To be improved if needed.
NEG_CUES = re.compile(r"\b(niega|niega[n]?|sin|no|descarta|descartan)\b", re.I)

NEG_CUES = re.compile(
    r"\b("
    r"niega(?:n)?|"
    r"sin|"
    r"descarta(?:n)?|"
    r"no\s+(?:refiere|presenta|consta|evidencia|hay|ha|han|tuvo|tiene|presentó|refirió)"
    r")\b",
    re.I
)

# Temporal cues
"""CHILD_CUES = re.compile(
    r"\b("
    # life stages
    r"(?:en|desde|durante)\s+(?:la|su)\s+(?:infancia|niñez|adolescencia)|"
    # "when (s)he was a child/minor"
    r"cuando\s+era\s+(?:niño|niña|menor|adolescente)|"
    # explicit "minor"
    r"menor\s+de\s+edad|"
    # explicit age 0-17: "a los 7 años", "con 15 años"
    r"(?:a\s+los|con)\s+(?:[0-9]|1[0-7])\s+a[nñ]os"
    r")\b",
    re.I
)"""

CHILD_CUES = re.compile(
    r"\b("
    # Life stages
    r"(?:en|desde|durante)\s+(?:la|su)\s+"
    r"(?:infancia|niñez|adolescencia|infancia\s+temprana|infancia\s+precoz)|"

    # When he/she was...
    r"cuando\s+era\s+(?:niñ[oa]|menor|adolescente|pequeñ[oa])|"

    # As a child / when young
    r"de\s+(?:niñ[oa]|pequeñ[oa])|"
    r"desde\s+(?:niñ[oa]|pequeñ[oa])|"
    r"siendo\s+(?:niñ[oa]|menor|adolescente|pequeñ[oa])|"

    # Explicit minor
    r"menor\s+de\s+edad|"

    # Explicit age 0-17
    r"(?:a\s+los|con)\s+(?:[0-9]|1[0-7])\s+a[nñ]os|"

    # Adolescence variants
    r"en\s+(?:su\s+)?adolescencia|"
    r"durante\s+la\s+adolescencia|"

    # Infancy variants without article rigidity
    r"en\s+(?:su\s+)?infancia|"
    r"durante\s+(?:la|su)\s+niñez|"
    r"desde\s+(?:la|su)\s+infancia"
    r")\b",
    re.I
)

RECENT_CUES = re.compile(
    r"\b("
    # relative durations
    r"(?:desde\s+hace|hace)\s+\d+\s+(?:a[nñ]os|mes(?:es)?|semanas|d[ií]as|horas)|"
    # "last N days/weeks/months/years"
    r"en\s+los\s+[uú]ltimos\s+\d+\s+(?:d[ií]as|semanas|mes(?:es)?|a[nñ]os)|"
    # adverbs
    r"recientemente|actualmente|en\s+la\s+actualidad|"
    # "last week/month/year", "this week/month/year"
    r"(?:la|este|esta)\s+(?:semana|mes|a[nñ]o)"
    r")\b",
    re.I
)

# Perpretator cues
PERPETRATOR_LINK_CUES = re.compile(
    r"\b("
    r"por\s+parte\s+de(?:l|la|los|las)?|"
    r"a\s+manos\s+de(?:l|la|los|las)?|"
    r"ejercid[oa]\s+por|"
    r"causad[oa]\s+por|"
    r"cometid[oa]\s+por|"
    r"perpetrad[oa]\s+por"
    r")\b",
    re.I
)

PERPETRATOR_CUES = re.compile(
    r"\b("
    r"padre\s+de\s+su\s+hij[oa]|"
    r"madre\s+de\s+su\s+hij[oa]|"
    r"padre|madre|"
    r"herman[oa]|"
    r"t[ií][oa]|"
    r"abuel[oa]|"
    r"prim[oa]|"
    r"profesor|profesora|"
    r"maestro|maestra"
    r")\b",
    re.I
)

<>:18: SyntaxWarning: invalid escape sequence '\s'
<>:18: SyntaxWarning: invalid escape sequence '\s'
C:\Users\adeli\AppData\Local\Temp\ipykernel_62368\1430133663.py:18: SyntaxWarning: invalid escape sequence '\s'
  r"(?:en|desde|durante)\s+(?:la|su)\s+(?:infancia|niñez|adolescencia)|"


In [255]:
CLAUSE_BREAKS = re.compile(r"[,:;]|\b(?:y|pero|aunque|sin embargo)\b", re.I)

##### Functions

In [ ]:
"""compiled_patterns = {
    k: re.compile("|".join(map(re.escape, v)), re.IGNORECASE)
    for k, v in ace_terms.items()
}"""

In [ ]:
"""def extract_terms(text, terms_dict):
    
    results = {}

    for category, terms in terms_dict.items():
        results[category] = [t for t in terms if t in text]

    return results"""

In [272]:
def cut_left_at_break(text):
    matches = list(CLAUSE_BREAKS.finditer(text))
    return text[matches[-1].end():] if matches else text

def cut_right_at_break(text):
    m = CLAUSE_BREAKS.search(text)
    return text[:m.start()] if m else text

In [273]:
def temporal_flags_local(span, left_window=40, right_window=40):
    text = span.doc.text

    L = max(span.start_char - left_window, 0)
    R = min(span.end_char + right_window, len(text))

    ctx_left = text[L:span.start_char]
    ctx_right = text[span.end_char:R]

    # keep only nearest local fragment
    ctx_left = cut_left_at_break(ctx_left)
    ctx_right = cut_right_at_break(ctx_right)

    child_match = CHILD_CUES.search(ctx_left) or CHILD_CUES.search(ctx_right)
    recent_match = RECENT_CUES.search(ctx_left) or RECENT_CUES.search(ctx_right)

    childhood = bool(child_match)
    recent = bool(recent_match)

    childhood_term = child_match.group(0) if child_match else None
    recent_term = recent_match.group(0) if recent_match else None

    return childhood, childhood_term, recent, recent_term

In [274]:
def perpetrator_flags_local(span, left_window=80, right_window=30):
    text = span.doc.text

    L = max(span.start_char - left_window, 0)
    R = min(span.end_char + right_window, len(text))

    ctx_left = text[L:span.start_char]
    ctx_right = text[span.end_char:R]

    # Restrict to nearest clause
    ctx_left = cut_left_at_break(ctx_left)
    # print("left: ", ctx_left)
    ctx_right = cut_right_at_break(ctx_right)
    # print("right: ", ctx_right)

    # For perpetrator attribution, left context is usually the important one
    left_term = PERPETRATOR_CUES.search(ctx_left)
    left_link = PERPETRATOR_LINK_CUES.search(ctx_left)

    right_term = PERPETRATOR_CUES.search(ctx_right)
    right_link = PERPETRATOR_LINK_CUES.search(ctx_right)

    # Strong rule: require both a perpetrator term and a link cue
    if left_term and left_link:
        return True, left_term.group(0)

    if right_term and right_link:
        return True, right_term.group(0)

    return False, None

In [275]:
# Previous version
"""def temporal_flags_sentence(span):
    sent = span.sent.text
    rel_start = span.start_char - span.sent.start_char
    rel_end   = span.end_char   - span.sent.start_char
    ctx = sent[:rel_start] + " " + sent[rel_end:]

    childhood = bool(CHILD_CUES.search(ctx))
    recent    = bool(RECENT_CUES.search(ctx))
    return childhood, recent"""

'def temporal_flags_sentence(span):\n    sent = span.sent.text\n    rel_start = span.start_char - span.sent.start_char\n    rel_end   = span.end_char   - span.sent.start_char\n    ctx = sent[:rel_start] + " " + sent[rel_end:]\n\n    childhood = bool(CHILD_CUES.search(ctx))\n    recent    = bool(RECENT_CUES.search(ctx))\n    return childhood, recent'

In [308]:
def annotate_ace(text: str, matcher):
    doc = nlp(text)
    matches = matcher(doc)

    ann = []
    for match_id, start, end in matches:
        span = doc[start:end]

        # Sentence-bounded context (left/right of the matched span)
        sent = span.sent.text
        rel_start = span.start_char - span.sent.start_char
        rel_end   = span.end_char   - span.sent.start_char
        sent_left  = sent[:rel_start]
        sent_right = sent[rel_end:]

        # Negation cues: check both sides (often left is enough, but this is safer)
        negated = bool(NEG_CUES.search(sent_left) or NEG_CUES.search(sent_right))

        # Temporal cues split into childhood vs recent
        # childhood, recent = temporal_flags_sentence(span)
        childhood, childhood_term, recent, recent_term = temporal_flags_local(
    span, left_window=40, right_window=40)
        
        # Perpetrator cues
        perpetrator_cue, perpetrator_term = perpetrator_flags_local(span, left_window=50, right_window=50)
        ann.append({
            "concept_id": nlp.vocab.strings[match_id],  # category or could be SNOMED code
            "span": span.text,
            "start_char": span.start_char,
            "end_char": span.end_char,
            "negated": negated,
            "childhood_cue": childhood,
            "childhood_term": childhood_term,
            "recent_cue": recent,
            "recent_term": recent_term,
            "perpetrator_cue": perpetrator_cue,
            "perpetrator_term": perpetrator_term,
        })

    # Deduplicate exact spans (keep first)
    ann = sorted(ann, key=lambda x: (x["start_char"], -x["end_char"]))
    out, seen = [], set()
    for a in ann:
        key = (a["concept_id"], a["start_char"], a["end_char"])
        if key not in seen:
            out.append(a)
            seen.add(key)

    return out

In [ ]:
def annotate_ace_2(text: str):
    doc = nlp(text)
    matches = matcher(doc)

    ann = []
    for match_id, start, end in matches:
        span = doc[start:end]

        negated = negation_flags_local(span, left_window=30, right_window=15)
        childhood, recent = temporal_flags_local(span, left_window=40, right_window=40)
        perpetrator_cue, perpetrator_term = perpetrator_flags_local(span, left_window=50, right_window=50)

        ann.append({
            "concept_id": nlp.vocab.strings[match_id],
            "span": span.text,
            "start_char": span.start_char,
            "end_char": span.end_char,
            "negated": negated,
            "childhood_cue": childhood,
            "recent_cue": recent,
            "perpetrator_cue": perpetrator_cue,
            "perpetrator_term": perpetrator_term,
        })

    ann = sorted(ann, key=lambda x: (x["start_char"], -x["end_char"]))
    out, seen = [], set()
    for a in ann:
        key = (a["concept_id"], a["start_char"], a["end_char"])
        if key not in seen:
            out.append(a)
            seen.add(key)

    return out

In [278]:
# Function that takes out accents, maintains "ñ"
def strip_accents_keep_enye(text):
    result = []
    for char in text:
        if char in ("ñ", "Ñ"):
            result.append(char)
        else:
            decomposed = unicodedata.normalize("NFD", char)
            base = "".join(c for c in decomposed if unicodedata.category(c) != "Mn")
            result.append(base)
    return "".join(result)

In [279]:
"""def strip_accents_keep_enye(text: str) -> str:
    out = []
    for ch in text:
        if ch in ("ñ", "Ñ"):
            out.append(ch)
        else:
            decomp = unicodedata.normalize("NFD", ch)
            base = "".join(c for c in decomp if unicodedata.category(c) != "Mn")
            out.append(base)
    return "".join(out)"""

'def strip_accents_keep_enye(text: str) -> str:\n    out = []\n    for ch in text:\n        if ch in ("ñ", "Ñ"):\n            out.append(ch)\n        else:\n            decomp = unicodedata.normalize("NFD", ch)\n            base = "".join(c for c in decomp if unicodedata.category(c) != "Mn")\n            out.append(base)\n    return "".join(out)'

In [280]:
# Does not work with PhraseMatcher
# Add to Spacy tokens the normalization that keeps "ñ", takes out accents
"""Token.set_extension(
    "noacc",
    getter=lambda token: strip_accents_keep_enye(token.text).lower(),
    force=True
)"""

'Token.set_extension(\n    "noacc",\n    getter=lambda token: strip_accents_keep_enye(token.text).lower(),\n    force=True\n)'

In [309]:
def annotate_notes_df(df_in, text_column, note_id_column, cols_order, matcher):
    all_ann = []

    for _, row in df_in.iterrows():
        note_text = row[text_column]

        # Skip missing / empty notes
        if pd.isna(note_text) or not str(note_text).strip():
            continue

        ann = annotate_ace(str(note_text), matcher)

        # If no annotations, skip
        if not ann:
            continue

        ann_df = pd.DataFrame(ann)
        ann_df["case_id"] = row[note_id_column]
        ann_df["case_history"] = note_text
        

        # Keep only desired columns in desired order
        ann_df = ann_df[cols_order]
        all_ann.append(ann_df)

    if all_ann:
        return pd.concat(all_ann, ignore_index=True)
    else:
        return pd.DataFrame(columns=cols_order)

In [307]:
def build_ace_matcher(nlp, ace_terms):
    matcher = PhraseMatcher(nlp.vocab, attr="LOWER")

    for concept_id, terms in ace_terms.items():
        pattern_texts = set()

        if not isinstance(terms, list):
            continue

        for term in terms:
            if not isinstance(term, str):
                continue

            term = term.strip()
            if not term:
                continue

            pattern_texts.add(term.lower())
            pattern_texts.add(strip_accents_keep_enye(term).lower())

        if pattern_texts:
            patterns = [nlp.make_doc(t) for t in pattern_texts]
            matcher.add(concept_id, patterns)

    return matcher

### First try with Spacy PhraseMatcher

In [302]:
# Load the dictionary of ACE terms
with open("ace_terms_dictionary.json", "r", encoding="utf-8") as f:
    ace_terms = json.load(f)

In [ ]:
# Third version.
nlp = spacy.load("es_core_news_md", disable=["ner", "parser", "tagger"])
nlp.add_pipe("sentencizer")

matcher = build_ace_matcher(nlp, ace_terms)


In [306]:
# Example
if __name__ == "__main__":
    note = "Niega maltrato físico. Refiere abuso sexual en la infancia y violencia doméstica en casa."
    print(annotate_ace(note))
    

[{'concept_id': 'ACE_PHYSICAL_ABUSE', 'span': 'maltrato físico', 'start_char': 6, 'end_char': 21, 'negated': True, 'childhood_cue': True, 'childhood_term': 'en la infancia', 'recent_cue': False, 'recent_term': None, 'perpetrator_cue': False, 'perpetrator_term': None}, {'concept_id': 'ACE_GENERAL_TRAUMA', 'span': 'maltrato', 'start_char': 6, 'end_char': 14, 'negated': True, 'childhood_cue': False, 'childhood_term': None, 'recent_cue': False, 'recent_term': None, 'perpetrator_cue': False, 'perpetrator_term': None}, {'concept_id': 'ACE_SEXUAL_ABUSE', 'span': 'abuso sexual', 'start_char': 31, 'end_char': 43, 'negated': False, 'childhood_cue': True, 'childhood_term': 'en la infancia', 'recent_cue': False, 'recent_term': None, 'perpetrator_cue': False, 'perpetrator_term': None}, {'concept_id': 'ACE_DOMESTIC_VIOLENCE_EXPOSURE', 'span': 'violencia doméstica', 'start_char': 61, 'end_char': 80, 'negated': False, 'childhood_cue': False, 'childhood_term': None, 'recent_cue': False, 'recent_term': 

In [243]:
note2= "abuso de alcohol que relaciona con haber sido victima de malos tratos por parte del padre de su hijo, y abusos sexuales en la infancia."
annotate_ace(note2)

[{'concept_id': 'ACE_GENERAL_TRAUMA',
  'span': 'victima',
  'start_char': 46,
  'end_char': 53,
  'negated': False,
  'childhood_cue': False,
  'childhood_term': None,
  'recent_cue': False,
  'recent_term': None,
  'perpetrator_cue': True,
  'perpetrator_term': 'padre de su hijo'},
 {'concept_id': 'ACE_PHYSICAL_ABUSE',
  'span': 'malos tratos',
  'start_char': 57,
  'end_char': 69,
  'negated': False,
  'childhood_cue': False,
  'childhood_term': None,
  'recent_cue': False,
  'recent_term': None,
  'perpetrator_cue': True,
  'perpetrator_term': 'padre de su hijo'},
 {'concept_id': 'ACE_SEXUAL_ABUSE',
  'span': 'abusos sexuales',
  'start_char': 104,
  'end_char': 119,
  'negated': False,
  'childhood_cue': True,
  'childhood_term': 'en la infancia',
  'recent_cue': False,
  'recent_term': None,
  'perpetrator_cue': False,
  'perpetrator_term': None}]

##### Load abusos file

In [ ]:
# Recover the path
directory_path = os.getcwd()
# root_path = os.path.dirname(directory_path)
print(directory_path)
# print(root_path)

c:\Users\adeli\Documents 4-Q2\Practicas-ACE\anotacionACE\ACE Project
c:\Users\adeli\Documents 4-Q2\Practicas-ACE\anotacionACE


In [75]:
# Try with patients in the file adversidad_precoz.xlsx (confirmed cases)
ace_cases_filepath = os.path.join(directory_path, 'data\\', "ace_confirmed_cases_eng.xlsx")
print(ace_cases_filepath)
ace_cases_df = pd.read_excel(ace_cases_filepath, na_values=["#NULL!"])

c:\Users\adeli\Documents 4-Q2\Practicas-ACE\anotacionACE\ACE Project\data\ace_confirmed_cases_eng.xlsx


In [76]:
len(ace_cases_df)

89

In [ ]:
ace_cases_df.head()

In [77]:
# Select a record
test_case = ace_cases_df.iloc[0]
note_ace = test_case["Psychiatric_History"]

In [200]:
note_ace

'Primer contacto con psiquiatría en el 2008, cuando tuvo varios ingresos por intoxicaciones etilicas y abuso de alcohol que relaciona con haber sido victima de malos tratos por parte del padre de su hijo, y abusos sexuales en la infancia.\nRealizó desentoxicacion en el Centro Los Almendros, por un año. En el 2012 empezó seguimiento en San Martin de Valdeiglesias. \nEstuvo ingresada en UDA en Noviembre 2016, 1 mes.\nUn ingreso psiquiátrico en el HRJC en agosto de 2016.\n\nTóxicos:- Alcohol:consumo perjudicial sobre los 25 años, de tipo dipsomaniaco. Niega periodos de consumo diario. Incremento de consumo en el 2008, en contexto de maltratos. Luego abandonó y se ha mantenido abstinente. Refiere en la actualidad consumo ocasional.\n- Cocaína y cannabis: niega consumo a diario pero reconoce consumo varias veces por semana\n- No otros tóxicos.\n\nSB: Natural de España. Madre alcoholica, desde pequeña se quedaba con sus vecinos, que luego acabaron solicitando la tutela. Vivió con sus padres 

In [244]:
annotate_ace(note_ace)

[{'concept_id': 'ACE_GENERAL_TRAUMA',
  'span': 'victima',
  'start_char': 148,
  'end_char': 155,
  'negated': False,
  'childhood_cue': False,
  'childhood_term': None,
  'recent_cue': False,
  'recent_term': None,
  'perpetrator_cue': True,
  'perpetrator_term': 'padre de su hijo'},
 {'concept_id': 'ACE_PHYSICAL_ABUSE',
  'span': 'malos tratos',
  'start_char': 159,
  'end_char': 171,
  'negated': False,
  'childhood_cue': False,
  'childhood_term': None,
  'recent_cue': False,
  'recent_term': None,
  'perpetrator_cue': True,
  'perpetrator_term': 'padre de su hijo'},
 {'concept_id': 'ACE_SEXUAL_ABUSE',
  'span': 'abusos sexuales',
  'start_char': 206,
  'end_char': 221,
  'negated': False,
  'childhood_cue': True,
  'childhood_term': 'en la infancia',
  'recent_cue': False,
  'recent_term': None,
  'perpetrator_cue': False,
  'perpetrator_term': None},
 {'concept_id': 'ACE_PARENT_SUBSTANCE',
  'span': 'Madre alcoholica',
  'start_char': 867,
  'end_char': 883,
  'negated': False,


In [245]:
df = pd.DataFrame(annotate_ace(note_ace))
df["note_id"] = test_case['Anonymized_Temporal']
cols_order=['note_id','concept_id', 'span', 'start_char', 'end_char', 'negated',
       'childhood_cue', 'childhood_term', 'recent_cue', 'recent_term',
       'perpetrator_cue', 'perpetrator_term', ]
df=df[cols_order]

In [246]:
df

,note_id,concept_id,span,start_char,end_char,negated,childhood_cue,childhood_term,recent_cue,recent_term,perpetrator_cue,perpetrator_term
0,216535,ACE_GENERAL_TRAUMA,victima,148,155,False,False,None,False,None,True,padre de su hijo
1,216535,ACE_PHYSICAL_ABUSE,malos tratos,159,171,False,False,None,False,None,True,padre de su hijo
2,216535,ACE_SEXUAL_ABUSE,abusos sexuales,206,221,False,True,en la infancia,False,None,False,None
3,216535,ACE_PARENT_SUBSTANCE,Madre alcoholica,867,883,False,False,None,False,None,False,None
4,216535,ACE_UNSPECIFIED_NEGLECT,negligencia,1102,1113,False,False,None,False,None,False,None


In [247]:
# Test another record
# Select a record
test_case_1 = ace_cases_df.iloc[1]
note_ace_1 = test_case_1["Psychiatric_History"]
print("Record: ", note_ace_1)
annotate_ace(note_ace_1)

Record:  - Inicio de síntomas psicóticos (ideación persecutoria, alucinaciones auditivas y visuales), según refiere, a los 12 años aproximadamente. 
- 1ª atención psiquiátrica a los 14 años (Clínica la Zarzuela). Pautan tratamiento con risperidona que toma durante 2 meses aproximadamente. Tras sobreingesta por accidente doméstico abandona el tratamiento.
- Vista en una ocasión a los 16 años en CSM Villalba (Dra. Sánchez) quien propone ingreso para observación y diagnóstico, pero la familia rechaza esta opción. 
- Consumo de drogas durante la adolescencia con criterios de abuso de cocaína y cannabis y dependencia a alcohol.  Abstinencia de tóxicos desde hace mas de 10 años.
- Inicia tratamiento a los 21 años con psiquiatra privado (Dr. Ramón Campos),  quien diagnostica a la paciente de desdoblamiento de la personalidad con esquizotipia y pauta tratamiento con citalopram, risperidona, reboxetina y clonazepam.
- Primer ingreso psiquiátrico en febrero 2009 en la Clínica San Juan de Dios co

[{'concept_id': 'ACE_GENERAL_TRAUMA',
  'span': 'maltrato',
  'start_char': 1894,
  'end_char': 1902,
  'negated': False,
  'childhood_cue': True,
  'childhood_term': 'en la infancia',
  'recent_cue': False,
  'recent_term': None,
  'perpetrator_cue': False,
  'perpetrator_term': None}]

In [253]:
# Test another record
# Select a record
test_case_2 = ace_cases_df.iloc[2]
note_ace_2 = test_case_2["Psychiatric_History"]
print("Record: ", note_ace_2)
annotate_ace(note_ace_2)

Record:  SBasal: 1/3 hermanos. Tiene dos hijos de padres distintos: una hija de 4 años (con 7m se separa del padre, problemas con él en cuanto a la hija, refiere maltrato "le absolvieron a pesar de partes médicos del hospital, me tiró por las escaleras"), otro de 8 años (le ve en vacaciones, verano, semana santa, los fds...custodia del padre). Todas las custodias a cargo de ellos por temas económicos. Mantiene relación de pareja, llevan dos años, viven juntos, bien. Él tiene dos hijos de su anterior pareja tmb. Padres separados con 6 años. Muchos problemas con la madre (probable TLP?), los otros hermanos han dejado de tener contacto con ella. Madre pasó CA de mama en 2021, vive en casa de la hermana en Vallecas (se vino tras la qx). Con el hermano no se habla "por mi excuñada". Hermano pasó CA, dice que la mujer de él "denunció a mi hermano, decía que le pegaba, que lo del CA era mentira..."...el hermano le ha dicho a la paciente ahora "que si no estoy yo están mejor, está mejor con el

[{'concept_id': 'ACE_GENERAL_TRAUMA',
  'span': 'maltrato',
  'start_char': 153,
  'end_char': 161,
  'negated': False,
  'childhood_cue': False,
  'childhood_term': None,
  'recent_cue': False,
  'recent_term': None,
  'perpetrator_cue': False,
  'perpetrator_term': None},
 {'concept_id': 'ACE_PARENT_DIVORCE_SEPARATION',
  'span': 'Padres separados',
  'start_char': 508,
  'end_char': 524,
  'negated': False,
  'childhood_cue': True,
  'childhood_term': 'con 6 años',
  'recent_cue': False,
  'recent_term': None,
  'perpetrator_cue': False,
  'perpetrator_term': None}]

In [252]:
# Test another record
# Select a record
test_case_3 = ace_cases_df.iloc[3]
note_ace_3 = test_case_3["Psychiatric_History"]
print("Record: ", note_ace_3)
annotate_ace(note_ace_3)

Record:  ANTEC MEDICOS
En estudio por dolor de mama, antec de CA de mama en la familia

Colecistectomizada

NAMC

TOXICOS
Niega consumo de tóxicos

ANTEC PSIQ
No 

ANTEC PSIQ FAM
Padre: alcoholismo

SB
Natural de Paraguay, 4 años en España, familia de origen Separada tras unos 15 años en pareja, tres hijos en Praguay. Trabaja como cuidadora de niños, pasa la semana en casa de sus empleadores, el fin de semana en una habitacion. Padres separados durante su infancia, ambos con nuevas parejas, rompió la relacion con el padre tras la separacion, hermanos por parte de padre que no conoce, ha conocido a uno aqui en España, con una mala experiencia (intento abusar de ella). Familia de origen humilde, trabajó desde los 7 años. Refiere maltrato por parte del padre de niña. Abusos sex por parte de marido de tia en una ocasion, pasó una temporada viviendo con ellos.


[{'concept_id': 'ACE_PARENT_DIVORCE_SEPARATION',
  'span': 'Padres separados',
  'start_char': 423,
  'end_char': 439,
  'negated': False,
  'childhood_cue': True,
  'childhood_term': 'durante su infancia',
  'recent_cue': False,
  'recent_term': None,
  'perpetrator_cue': False,
  'perpetrator_term': None},
 {'concept_id': 'ACE_GENERAL_TRAUMA',
  'span': 'maltrato',
  'start_char': 728,
  'end_char': 736,
  'negated': False,
  'childhood_cue': True,
  'childhood_term': 'de niña',
  'recent_cue': False,
  'recent_term': None,
  'perpetrator_cue': True,
  'perpetrator_term': 'padre'}]

### Apply over all the dataframe the annotation function

In [295]:
cols_order = [
    "case_id",
    "concept_id",
    "span",
    "start_char",
    "end_char",
    "negated",
    "childhood_cue",
    "childhood_term",
    "recent_cue",
    "recent_term",
    "perpetrator_cue",
    "perpetrator_term",
    "case_history"
]

In [296]:
# Choose the id column to use and the text column to annotate
case_history_col = "Psychiatric_History"
case_id_col = "Anonymized_Temporal"

In [311]:
annotations_df = annotate_notes_df(
    ace_cases_df,
    text_column= case_history_col,
    note_id_column= case_id_col, 
    cols_order= cols_order, 
    matcher=matcher
)

In [312]:
annotations_df.drop(columns="case_history")

,case_id,concept_id,span,start_char,end_char,negated,childhood_cue,childhood_term,recent_cue,recent_term,perpetrator_cue,perpetrator_term
0,216535,ACE_GENERAL_TRAUMA,victima,148,155,False,False,None,False,None,True,padre de su hijo
1,216535,ACE_PHYSICAL_ABUSE,malos tratos,159,171,False,False,None,False,None,True,padre de su hijo
2,216535,ACE_SEXUAL_ABUSE,abusos sexuales,206,221,False,True,en la infancia,False,None,False,None
3,216535,ACE_PARENT_SUBSTANCE,Madre alcoholica,867,883,False,False,None,False,None,False,None
4,216535,ACE_UNSPECIFIED_NEGLECT,negligencia,1102,1113,False,False,None,False,None,False,None
5,232165,ACE_GENERAL_TRAUMA,maltrato,1894,1902,False,True,en la infancia,False,None,False,None
6,262678,ACE_GENERAL_TRAUMA,maltrato,153,161,False,False,None,False,None,False,None
7,262678,ACE_PARENT_DIVORCE_SEPARATION,Padres separados,508,524,False,True,con 6 años,False,None,False,None
8,269487,ACE_PARENT_DIVORCE_SEPARATION,Padres separados,423,439,False,True,durante su infancia,False,None,False,None
9,269487,ACE_GENERAL_TRAUMA,maltrato,728,736,False,True,de niña,False,None,True,padre
